# Customer Churn Prediction — Machine Learning

**Objectif :** analyser et prédire la résiliation de clients télécoms.

**Données :** [Telco Customer Churn (Kaggle)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn).

Ce notebook est prêt à exécuter : les résultats seront produits avec le CSV. Les scores ne sont pas inventés.

## 1. Bibliothèques

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, classification_report,
                             ConfusionMatrixDisplay)
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 2. Chargement des données

Place le CSV dans `data/`, ou téléverse-le dans Google Colab quand la fenêtre apparaît.

In [ ]:
filename = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
path = next((p for p in [Path("data") / filename, Path(filename)] if p.exists()), None)
if path is None:
    try:
        from google.colab import files
        files.upload()
        path = Path(filename)
    except ImportError:
        raise FileNotFoundError("Télécharge le CSV sur Kaggle et place-le dans data/.")
df = pd.read_csv(path)
print("Dimensions :", df.shape)
display(df.head())

## 3. Contrôle de qualité

In [ ]:
print("Doublons :", df.duplicated().sum())
print("Identifiants dupliqués :", df.customerID.duplicated().sum())
print("Churn :")
display(df.Churn.value_counts().to_frame("Nombre"))
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("TotalCharges manquants :", df.TotalCharges.isna().sum())
# Les valeurs manquantes seront imputées uniquement à partir du train.

## 4. Analyse exploratoire

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Churn", ax=axes[0])
axes[0].set_title("Répartition du churn")
rates = df.groupby("Contract")["Churn"].apply(lambda s: 100*(s=="Yes").mean())
sns.barplot(x=rates.index, y=rates.values, ax=axes[1])
axes[1].set_title("Taux de churn par contrat")
axes[1].set_ylabel("Churn (%)")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()
display(rates.round(2).to_frame("Churn (%)"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="Churn", y="tenure", ax=axes[0])
axes[0].set_title("Ancienneté")
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
axes[1].set_title("Frais mensuels")
plt.tight_layout(); plt.show()

## 5. Prétraitement et séparation

60 % entraînement, 20 % validation, 20 % test. L’imputation, l’encodage et la normalisation sont intégrés au pipeline pour éviter les fuites de données.

In [ ]:
X = df.drop(columns=["customerID", "Churn"])
y = df.Churn.map({"No": 0, "Yes": 1}).astype(int)
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=.25, stratify=y_trainval, random_state=RANDOM_STATE)
num = X.select_dtypes(include="number").columns.tolist()
cat = X.select_dtypes(exclude="number").columns.tolist()
preprocessing = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), cat)
])
print("Train / Validation / Test :", len(X_train), len(X_val), len(X_test))

## 6. Entraînement des modèles et comparaison sur validation

In [ ]:
estimators = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)
}
def evaluate(y_true, proba, threshold=.5):
    pred = (proba >= threshold).astype(int)
    return dict(Accuracy=accuracy_score(y_true,pred), Precision=precision_score(y_true,pred,zero_division=0),
                Recall=recall_score(y_true,pred,zero_division=0), F1=f1_score(y_true,pred,zero_division=0),
                ROC_AUC=roc_auc_score(y_true,proba))
pipelines, val_prob = {}, {}
for name, model in estimators.items():
    pipe = Pipeline([("preprocessing", preprocessing), ("classifier", model)])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe
    val_prob[name] = pipe.predict_proba(X_val)[:, 1]
display(pd.DataFrame({name:evaluate(y_val, p) for name,p in val_prob.items()}).T.round(4))

## 7. Choix du modèle et du seuil (validation uniquement)

In [ ]:
rows=[]
for name, p in val_prob.items():
    for threshold in np.round(np.arange(.10,.901,.01),2):
        rows.append({"Model":name,"Threshold":threshold,**evaluate(y_val,p,threshold)})
selection = pd.DataFrame(rows).sort_values(["F1","ROC_AUC","Model"],ascending=[False,False,True]).iloc[0]
chosen_name, chosen_threshold = selection.Model, float(selection.Threshold)
print("Modèle :", chosen_name, "| Seuil :", chosen_threshold)
sub = pd.DataFrame(rows).query("Model == @chosen_name")
for metric in ["Precision","Recall","F1"]:
    plt.plot(sub.Threshold,sub[metric],label=metric)
plt.axvline(chosen_threshold,linestyle="--",color="gray")
plt.xlabel("Seuil"); plt.ylabel("Score de validation"); plt.legend(); plt.title("Choix du seuil")
plt.show()

## 8. Évaluation finale sur le jeu de test

In [ ]:
prob_test = pipelines[chosen_name].predict_proba(X_test)[:,1]
pred_test = (prob_test >= chosen_threshold).astype(int)
final_scores = evaluate(y_test,prob_test,chosen_threshold)
display(pd.DataFrame([final_scores],index=[chosen_name]).round(4))
print(classification_report(y_test,pred_test,target_names=["No churn","Churn"],zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred_test,display_labels=["No churn","Churn"],cmap="Blues",colorbar=False)
plt.title("Matrice de confusion — Test");plt.show()
fpr,tpr,_ = roc_curve(y_test,prob_test)
plt.plot(fpr,tpr,label=f"{chosen_name} (AUC={final_scores['ROC_AUC']:.3f})")
plt.plot([0,1],[0,1],"--",color="gray")
plt.xlabel("False positive rate");plt.ylabel("True positive rate");plt.title("Courbe ROC — Test");plt.legend();plt.show()

## 9. Conclusion

Après exécution, indique le modèle sélectionné, le seuil retenu, les performances sur test et les limites du projet.

**Limites :** données pédagogiques, résultats non garantis en production ; association ne signifie pas causalité.

**Reproductibilité :** `random_state=42`, séparation train/validation/test, pipeline de prétraitement.